In [1]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

In [25]:
from icare_risk.data.dataset import Dataset
from icare_risk.core.schema import ClinicalSchema, TableSchema

# 1. Define your hospital's specific schema mapping
my_schema = ClinicalSchema(
    episodes=TableSchema("episodes",
        subject_col="SUBJECT",
        encounter_col="ENCNTR_ID",
        #encounter_col=None,
        timestamp_col="ADMISSION_DATE"
    ),
    vitals=TableSchema("vitals",
        timestamp_col="OBSERVATION_PERFORMED_DT",
        code_col="OBSERVATION_CODE",
        value_col="OBSERVATION_RESULT_CLEAN"
    ),
    problems=TableSchema("problems",
        timestamp_col="PROBLEM_DT_TM",
        code_col="PROBLEM_CODE"
    ),
    prescribing=TableSchema("prescribing",
        timestamp_col="ORDER_DT_TM",
        code_col="MEDICATION_NAME_SHORT",
        value_col="ORDERED_DOSE"
    ),
    pathology=TableSchema("pathology",
        timestamp_col="RESULT_AVAILABLE_DT",
        code_col="TEST_CODE",
        value_col="RESULT_CLEANED"
    )
)

path = '/app/data/mock/sirs_test'

# 2. Initialize the Dataset and map the files (Zero SQL!)
dataset = Dataset()
dataset.register_csv("episodes", f"{path}/episodes.csv")
dataset.register_csv("pathology", f"{path}/pathology.csv")
dataset.register_csv("problems", f"{path}/problems.csv")
dataset.register_csv("vitals", f"{path}/vitals.csv")

# 3. Get all episodes
all_episodes_df = dataset.get_episodes("episodes")
display(all_episodes_df.head(10))

,SUBJECT,ADMISSION_DATE,ADMISSION_TIME,DISCHARGE_DATE,ENCNTR_ID
0,101,2026-05-01,08:00:00,2026-06-01,stay_0
1,101,2026-09-01,08:00:00,2026-09-10,stay_1
2,102,2026-09-02,08:00:00,2026-09-10,stay_2
3,103,2026-09-03,08:00:00,2026-09-10,stay_3
4,104,2026-09-04,08:00:00,2026-09-10,stay_4
5,105,2026-09-05,08:00:00,2026-09-10,stay_5
6,10001,2023-01-01,08:00:00,2023-01-10,stay_6
7,10003,2023-01-01,08:00:00,2023-01-10,stay_7


In [26]:
from icare_risk.runner.builder import build_feature_matrix

# Import phenotypes to trigger their registration
import icare_risk.phenotypes.composite
import icare_risk.phenotypes.vitals

# 4. Generate the Feature Matrix
print(f"Building matrix for {len(all_episodes_df)} stays...")

feature_matrix = build_feature_matrix(
    cohort_df=all_episodes_df,
    dataset=dataset,
    schema=my_schema,
    phenotypes=["is_hypoxic_first_24h"] # Specify which features to run
)

print("\nFinal Feature Matrix:")
print(feature_matrix)


Building matrix for 8 stays...

Final Feature Matrix:
   SUBJECT ENCNTR_ID  is_hypoxic_first_24h
0      101    stay_0                 False
1      101    stay_1                 False
2      102    stay_2                 False
3      103    stay_3                 False
4      104    stay_4                 False
5      105    stay_5                 False
6    10001    stay_6                 False
7    10003    stay_7                 False
